### PV estimations

First step: inspect the raw PV ramp data before calculating any EMS values.

#### Load and inspect the raw PV data

This cell reads all PV ramp files, applies the same INA3 PV sensor correction as the larger analysis script, and plots the corrected PV voltage, current, and power. The goal is only to check if the data looks reasonable.

In [ ]:
from pathlib import Path
import re

import pandas as pd
import matplotlib.pyplot as plt


DATA_DIR = Path("../../data/PV_test/New_test")

# PV panel is measured by INA3 / address 0x44.
# These correction factors come from the sensor calibration.
PV_VOLTAGE_CORRECTION = lambda v: v - 0.180
PV_CURRENT_CORRECTION = lambda i: i + 0.000138


def get_distance_cm(file_name):
    match = re.search(r"PV_0?(\d+)cm", file_name)
    return int(match.group(1))


def load_pv_file(path):
    df = pd.read_csv(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    df["time_s"] = (
        df["timestamp"] - df["timestamp"].iloc[0]
    ).dt.total_seconds()

    df["pv_voltage_V"] = PV_VOLTAGE_CORRECTION(df["ina3_bus_V"])
    df["pv_current_A"] = PV_CURRENT_CORRECTION(df["ina3_current_mA"] / 1000)

    # Formula: P [mW] = V [V] * I [A] * 1000
    df["pv_power_mW"] = df["pv_voltage_V"] * df["pv_current_A"] * 1000

    return df


pv_files = sorted(DATA_DIR.glob("PV_*cm_ramp.csv"))

print("PV ramp files:")
for file in pv_files:
    print(" -", file.name)


pv_data = {}

for file in pv_files:
    distance_cm = get_distance_cm(file.name)
    df = load_pv_file(file)
    pv_data[distance_cm] = df

    print(f"\n{file.name}")
    print("Rows:", len(df))
    print("Time:", df["time_s"].min(), "to", df["time_s"].max(), "s")
    print("Voltage:", df["pv_voltage_V"].min(), "to", df["pv_voltage_V"].max(), "V")
    print("Current:", df["pv_current_A"].min(), "to", df["pv_current_A"].max(), "A")
    print("Power:", df["pv_power_mW"].min(), "to", df["pv_power_mW"].max(), "mW")


display(pv_data[1].head())

#### Plot the corrected raw PV curves

These plots are just for visual checking. The different distances represent different light intensities.

In [ ]:
plt.figure(figsize=(10, 5))

for distance_cm, df in pv_data.items():
    plt.plot(df["time_s"], df["pv_voltage_V"], label=f"{distance_cm} cm")

plt.xlabel("Time [s]")
plt.ylabel("PV voltage [V]")
plt.title("Corrected raw PV voltage")
plt.grid(True)
plt.legend()
plt.show()


plt.figure(figsize=(10, 5))

for distance_cm, df in pv_data.items():
    plt.plot(df["time_s"], df["pv_current_A"], label=f"{distance_cm} cm")

plt.xlabel("Time [s]")
plt.ylabel("PV current [A]")
plt.title("Corrected raw PV current")
plt.grid(True)
plt.legend()
plt.show()


plt.figure(figsize=(10, 5))

for distance_cm, df in pv_data.items():
    plt.plot(df["time_s"], df["pv_power_mW"], label=f"{distance_cm} cm")

plt.xlabel("Time [s]")
plt.ylabel("PV power [mW]")
plt.title("Corrected raw PV power")
plt.grid(True)
plt.legend()
plt.show()